# Assignment Final
Los datos que se incluyen en el fichero `datos_ventas_centros_comerciales.csv` representan casi 100.000 compra-ventas realizadas en un grupo de Centros Comerciales de Turquía durante un conjunto de años. 

Como analista del holding propietario de los centros comerciales, realiza un análisis respondiendo a las siguientes preguntas.  

## Notas y consejos

> **Consejo 1:** Es buena práctica, especialmente en preguntas que piden "el que más" o "el que menos" de algo,  
> considerar que puede haber **más de un valor máximo o mínimo** y decidir si devolverlo como **una colección de elementos** en lugar de un único valor.  

> **Consejo 2:** Solo es obligatorio crear una función cuando la pregunta lo indique explícitamente.  
> Sin embargo, te animo a crear funciones para organizar tu código y reutilizarlo cuando tenga sentido.  

> **Consejo 3:** Algunas preguntas se pueden resolver de manera muy similar, cambiando solo pequeños detalles del código.  
> Te invito a que, si notas que una solución se parece mucho a otra, pienses en **formas alternativas** de resolverla.

> **Consejo 4:** Antes de entregar, **reinicia el kernel y ejecuta todas las celdas** (`Restart Kernel + Run All`)  
> para asegurarte de que todo el código funciona correctamente desde cero.  
> Si alguna celda no se ejecuta correctamente, el ejercicio se calificará con un **máximo del 50%**.



### Definiciones de columnas del dataset

| Columna        | Descripción                                                                 |
|----------------|-----------------------------------------------------------------------------|
| invoice_no     | Número de factura. Una combinación de la letra 'I' y un número de 6 dígitos que identifica cada operación. |
| customer_id    | Número de cliente. Una combinación de la letra 'C' y un número de 6 dígitos que identifica a cada cliente. |
| category       | Categoría del producto comprado.                                            |
| quantity       | Cantidad de artículos comprados en la transacción.                          |
| price          | Precio unitario del producto en liras turcas (TL).                          |
| invoice_date   | Fecha en que se realizó la transacción.                                     |
| shopping_mall  | Nombre del centro comercial donde se realizó la compra.                     |

**Aclaración:** En las preguntas posteriores hay una diferencia importante entre **ingresos** y **número de ventas**.  
- **Ingresos**: se calculan multiplicando la cantidad (`quantity`) por el precio (`price`) de cada transacción.  
- **Número de ventas**: simplemente cuenta cuántas transacciones o facturas se realizaron, sin tener en cuenta el importe.









# 0. Exploración de Datos

Leer el CSV con pandas y explorar la información básica.


1. Cargar el CSV en una variable `datos`.


In [1]:
import pandas as pd

df_entregable_inicial = pd.read_csv("../datos/datos_ventas_centros_comerciales.csv")


2. Mostrar las primeras 5 filas y la información de las columnas.  


In [2]:
df_entregable_inicial.head(5)

,invoice_no,customer_id,category,quantity,price,invoice_date,shopping_mall
0,I138884,C241288,Clothing,5,8172.24,05-08-2022,Kanyon
1,I317333,C111565,Shoes,3,8868.64,12-12-2021,Forum Istanbul
2,I127801,C266599,Clothing,1,4863.95,09-11-2021,Metrocity
3,I173702,C988172,Shoes,5,711.48,16-05-2021,Metropol AVM
4,I337046,C189076,Books,4,1110.32,24-10-2021,Kanyon


3. Contar el número de filas y columnas.

In [3]:
filas, columnas = df_entregable_inicial.shape
print("Número de filas:",filas)
print("Número de columnas: ", columnas)

Número de filas: 99457
Número de columnas:  7


4. Mostrar los nombres de todas las columnas disponibles.  


In [4]:
print(df_entregable_inicial.columns)

Index(['invoice_no', 'customer_id', 'category', 'quantity', 'price',
       'invoice_date', 'shopping_mall'],
      dtype='object')


5. ¿Hay alguna columna con datos faltantes?  Si es así, ¿cuántos datos faltan por columna?  

Borra las filas que tengan algún dato faltante y vuelve a contar cuántas filas quedan.


In [5]:
faltantes = df_entregable_inicial.isnull().sum()
print("Valores faltantes por columna:")
print(faltantes)
df_entregable = df_entregable_inicial.dropna()
filas_sin_nulos = df_entregable.shape[0]
print("Filas después de eliminar faltantes:", filas_sin_nulos)

Valores faltantes por columna:
invoice_no       0
customer_id      0
category         0
quantity         0
price            0
invoice_date     0
shopping_mall    0
dtype: int64
Filas después de eliminar faltantes: 99457


6. Convertir `invoice_date` a tipo datetime y verificar que el cambio se realizó correctamente.


In [6]:
df_entregable["invoice_date"]= pd.to_datetime(df_entregable["invoice_date"], dayfirst=True)
print(df_entregable["invoice_date"].dtypes)

datetime64[ns]


7. Explora alguna otra característica del dataset que consideres relevante.  
   *Os dejo algunos ejemplos para que podáis inspiraros: categorías, centros comerciales o rangos de precios. Estos ejemplos no cuentan.*


In [7]:
facturas_repetidas = df_entregable[df_entregable['invoice_no'].duplicated(keep=False)]
print(facturas_repetidas)

Empty DataFrame
Columns: [invoice_no, customer_id, category, quantity, price, invoice_date, shopping_mall]
Index: []


# 1. ¿Cuál es el total de ingresos generados por todas las ventas registradas?

Usa una función y opcionalmente guardar en `src/utils.py`.

Guarda el resultado en una variable llamada `total_ingresos`



In [8]:
def calcular_total_ingresos(dataframe):
    """Calula el importe ventas totales creando una ueva columna de ventas (pxq)"""
    dataframe["ventas_en_€"] = dataframe["quantity"] * dataframe["price"]
    total_ingresos = dataframe["ventas_en_€"].sum()
    return total_ingresos
total_ingresos = calcular_total_ingresos(df_entregable)
print(f"El ingreso total de todas las ventas fue de '{total_ingresos}'")

El ingreso total de todas las ventas fue de '1490921515.71'


# 2. ¿Cuál es el centro comercial que ha generado más ventas?
Usa una función y opcionalmente guardar en `src/utils.py`.

Guarda el resultado en una variable llamada `centro_comercial_mas_ventas`


In [9]:
def centro_con_mas_ventas(dataframe):
    """ Devuelve elentro comercial con mayor ventas, medido como número de transeacciones"""
    ventas_por_centro = dataframe.groupby('shopping_mall')['invoice_no'].count()
    centro_mas_ventas = ventas_por_centro.idxmax()
    return centro_mas_ventas

centro_comercial_mas_ventas = centro_con_mas_ventas(df_entregable)
print(f"El centro comercial con más ventas fue el '{centro_comercial_mas_ventas}'")

El centro comercial con más ventas fue el 'Mall of Istanbul'


# 3. ¿Cuál es la categoría de producto que ha generado mas ventas? 
Guarda el resultado en una variable llamada `categoria_mas_vendida`


In [10]:
categoria_mas_vendida = df_entregable.groupby("category")["ventas_en_€"].sum().idxmax()
print(f"La categoría de producto con mas importe de venta es '{categoria_mas_vendida}'")

La categoría de producto con mas importe de venta es 'Clothing'


# 4. ¿Cuál es el producto más caro vendido?
Guarda el resultado en una variable llamada `categoria_producto_mas_caro`



In [11]:
categoria_producto_mas_caro = df_entregable.loc[df_entregable['price'].idxmax(), 'category']
print(f"La categoría del producto más caro es '{categoria_producto_mas_caro}'")

La categoría del producto más caro es 'Toys'


# 5. ¿Cuál es la factura más antigua registrada en el dataset?
Guarda el resultado en una variable llamada `factura_mas_antigua`.  

In [12]:
factura_mas_antigua = df_entregable.loc[df_entregable["invoice_date"].idxmin(),"invoice_no"]
print(f"La factuta más antigüa es la '{factura_mas_antigua}'")

La factuta más antigüa es la 'I301161'


# 6. ¿Cuál es la cantidad promedio de productos vendidos por transacción?
Guarda el resultado en una variable llamada `cantidad_promedio_por_transaccion`



In [13]:
cantidad_promedio_por_transaccion = df_entregable["quantity"].mean()
print(f"La cantidad promedio de productos por transacción es de '{cantidad_promedio_por_transaccion.round(2)}'")

La cantidad promedio de productos por transacción es de '3.0'


# 7. ¿Cuál es el día con más ventas registradas?
Guarda el resultado en una variable llamada `dia_con_mas_ventas`.

In [14]:
ventas_por_dia = df_entregable.groupby("invoice_date")["ventas_en_€"].sum()
dia_con_mas_ventas = ventas_por_dia.idxmax()
print(f"El día con más ventas registradas fue el '{dia_con_mas_ventas}'")


El día con más ventas registradas fue el '2022-01-07 00:00:00'


# 8. ¿Cuál es el cliente que más ha gastado en total?
Guarda el resultado en una variable llamada `cliente_que_mas_gasto`


In [15]:
gasto_por_cliente = df_entregable.groupby("customer_id")["ventas_en_€"].sum()
cliente_que_mas_gasto = gasto_por_cliente.idxmax()
print(f"El ID del cliente que más gasto es '{cliente_que_mas_gasto}'")

El ID del cliente que más gasto es 'C307063'


# 9. ¿Cuál es la factura con el precio total más alto?
Guarda el resultado en una variable llamada `factura_max_valor`


In [16]:
factura_max_valor = df_entregable.loc[df_entregable["ventas_en_€"].idxmax(),"invoice_no"]
print(f"El número de la factura con el precio total más alto es'{factura_max_valor}'")

El número de la factura con el precio total más alto es'I311640'


# 10. ¿Cuál es la distribución porcentual de ventas por categoría de producto?
Guarda el resultado en una variable llamada `distribucion_porcentual_ventas_por_categoria`. 

El tipo de `distribucion_porcentual_ventas_por_categoria` debe ser `pandas.core.series.Series` donde las etiquetas deben ser las categorias y los valores deben ser la distribucion porcentual de las ventas por categoría


In [17]:
ventas_por_categoria = df_entregable.groupby('category')['ventas_en_€'].sum()
distribucion_porcentual_ventas_por_categoria = (ventas_por_categoria / ventas_por_categoria.sum()) * 100
print("Distribución porcentual por categoría:")
print(distribucion_porcentual_ventas_por_categoria.round(2))
type(distribucion_porcentual_ventas_por_categoria)

Distribución porcentual por categoría:
category
Books               4.99
Clothing           34.50
Cosmetics          15.22
Food & Beverage    14.92
Shoes              10.17
Souvenir            4.97
Technology          5.06
Toys               10.16
Name: ventas_en_€, dtype: float64


pandas.core.series.Series

# 11. ¿Cuál es el día de la semana con más ventas registradas en cada centro comercial?
Guarda el resultado en una variable llamada `dia_semana_mas_ventas_por_centro`. 

El tipo de `dia_semana_mas_ventas_por_centro` debe ser `pandas.core.series.Series` donde las etiquetas deben ser el nombre de los centros comerciales y los valores deben ser el nombre del dia de la semana con mas ventas.

In [18]:
df_entregable['dia_semana'] = df_entregable['invoice_date'].dt.day_name()
ventas_centro_por_dia = df_entregable.groupby(['shopping_mall', 'dia_semana'])['invoice_no'].count()
dia_semana_mas_ventas_por_centro = ventas_centro_por_dia.groupby('shopping_mall').idxmax()
dia_semana_mas_ventas_por_centro = dia_semana_mas_ventas_por_centro.apply(lambda x: x[1])
dia_semana_mas_ventas_por_centro = pd.Series(dia_semana_mas_ventas_por_centro)

print("Día de la semana con más ventas por centro comercial:")
print(dia_semana_mas_ventas_por_centro)
print(type(dia_semana_mas_ventas_por_centro))

Día de la semana con más ventas por centro comercial:
shopping_mall
Cevahir AVM           Sunday
Emaar Square Mall     Friday
Forum Istanbul        Monday
Istinye Park          Friday
Kanyon                Monday
Mall of Istanbul     Tuesday
Metrocity             Monday
Metropol AVM         Tuesday
Viaport Outlet        Friday
Zorlu Center         Tuesday
Name: invoice_no, dtype: object
<class 'pandas.core.series.Series'>
